# Fine-tune 4B — YOLO26n-pose trên GPU T4

Notebook chạy độc lập cho phần còn thiếu của Lab 18: tải tiger-pose, fine-tune **đúng 40 epoch ở imgsz 640 trên CUDA**, đánh giá mAP trên 53 ảnh validation, tạo ảnh lỗi Q11 và cập nhật `submission.zip`.

**Yêu cầu:** chọn GPU T4 trước khi chạy (`Runtime → Change runtime type → T4 GPU`). Notebook cố ý dừng nếu không thấy CUDA; không có CPU fallback và không giảm epoch. Kết quả training thật sẽ thay số liệu 3 epoch CPU trong báo cáo.

Chạy `Runtime → Run all`. Ô cuối tải `submission.zip` xuống máy.

In [ ]:
%pip install -q "ultralytics==8.4.171"

In [ ]:
import os, sys, json, math, time, base64, shutil, csv
from pathlib import Path

import cv2
import numpy as np
import torch
import ultralytics
import yaml
from IPython.display import display, Image as IPyImage
from ultralytics import YOLO
from ultralytics.data.utils import check_det_dataset

assert torch.cuda.is_available(), (
    "Notebook này cần GPU CUDA (Colab: Runtime → Change runtime type → T4 GPU). "
    "Không chạy fine-tune CPU vì rubric yêu cầu 40 epoch trên GPU."
)
assert torch.cuda.device_count() >= 1
DEVICE = 0
print(f"GPU: {torch.cuda.get_device_name(0)} | torch {torch.__version__} | "
      f"CUDA {torch.version.cuda} | ultralytics {ultralytics.__version__}")
assert ultralytics.__version__ == "8.4.171", f"Sai ultralytics version: {ultralytics.__version__}"


In [ ]:
# Tải tiger-pose (210 train, 53 val), tải weights YOLO26n-pose và cấu hình flip_idx giải phẫu.
DATA = check_det_dataset("tiger-pose.yaml")
DATA_ROOT = Path(DATA["path"]).resolve()
assert len(list((DATA_ROOT / "images/train").glob("*.jpg"))) == 210
assert len(list((DATA_ROOT / "images/val").glob("*.jpg"))) == 53
KPT_NAMES = DATA["kpt_names"][0]
FLIP_IDX = [0, 1, 2, 3, 7, 6, 5, 4, 10, 11, 8, 9]
assert len(KPT_NAMES) == len(FLIP_IDX) == 12
assert sorted(FLIP_IDX) == list(range(12))

DATA_YAML = Path("tiger-pose-anat.yaml").resolve()
yaml.safe_dump({
    "path": str(DATA_ROOT), "train": "images/train", "val": "images/val",
    "kpt_shape": [12, 2], "flip_idx": FLIP_IDX,
    "names": {0: "tiger"}, "kpt_names": {0: KPT_NAMES},
}, DATA_YAML.open("w", encoding="utf-8"), sort_keys=False)
print(f"Dataset: {DATA_ROOT} | train=210 | val=53 | keypoints={KPT_NAMES}")
print(f"YAML: {DATA_YAML}")


In [ ]:
# Fine-tune bắt buộc: đúng 40 epoch, imgsz 640, GPU 0. Không có CPU fallback.
EPOCHS, IMGSZ = 40, 640
model = YOLO("yolo26n-pose.pt")
start = time.time()
train_result = model.train(
    data=str(DATA_YAML), epochs=40, imgsz=640, batch=16, device=0, workers=2,
    seed=0, patience=100, project="runs_4b", name="tiger_anat_40ep",
    exist_ok=True, verbose=False, plots=True,
)
TRAIN_MIN = (time.time() - start) / 60
SAVE_DIR = Path(model.trainer.save_dir).resolve()
RESULTS_CSV = SAVE_DIR / "results.csv"
with RESULTS_CSV.open(newline="", encoding="utf-8") as f:
    train_rows = list(csv.DictReader(f))
completed_epochs = len(train_rows)
assert completed_epochs == 40, f"Chỉ ghi nhận {completed_epochs}/40 epoch trong {RESULTS_CSV}"
print(f"Đã hoàn tất {completed_epochs}/40 epoch trên {torch.cuda.get_device_name(0)} trong {TRAIN_MIN:.1f} phút.")
print(f"Checkpoint tốt nhất: {SAVE_DIR / 'weights' / 'best.pt'}")


In [ ]:
# Đánh giá checkpoint tốt nhất trên toàn bộ 53 ảnh validation.
best = YOLO(str(SAVE_DIR / "weights" / "best.pt"))
metrics = best.val(data=str(DATA_YAML), imgsz=640, batch=16, device=0, verbose=False, plots=True)
BOX_MAP = float(metrics.box.map)
POSE_MAP50 = float(metrics.pose.map50)
POSE_MAP = float(metrics.pose.map)
RESULT_4B = {
    "flip_idx": "giải phẫu", "epochs": completed_epochs, "imgsz": 640,
    "Box mAP50-95": BOX_MAP, "Pose mAP50": POSE_MAP50,
    "Pose mAP50-95": POSE_MAP, "train (phút)": round(TRAIN_MIN, 1),
    "device": torch.cuda.get_device_name(0), "val_images": 53,
}
print(json.dumps(RESULT_4B, ensure_ascii=False, indent=2))


In [ ]:
# Phân tích từng ảnh val bằng OKS và dựng sáu ảnh tệ nhất cho Q11.
SIGMAS = np.full(12, 1 / 12, dtype=float)
val_imgs = sorted((DATA_ROOT / "images/val").glob("*.jpg"))
records = []
for img_path, pred_result in zip(
    val_imgs,
    best.predict([str(p) for p in val_imgs], imgsz=640, conf=0.25,
                 device=0, verbose=False, stream=True),
):
    image = cv2.imread(str(img_path))
    h, w = image.shape[:2]
    row = np.loadtxt(DATA_ROOT / "labels/val" / f"{img_path.stem}.txt", ndmin=2)[0]
    gt = row[5:].reshape(-1, 2) * [w, h]
    cx, cy, bw, bh = row[1:5] * [w, h, w, h]
    area = max(float(bw * bh) * 0.53, 1.0)
    record = {"path": img_path, "gt": gt, "pred": None, "oks": 0.0,
              "err_px": np.full(12, np.nan), "area": area}
    if len(pred_result.boxes):
        ix = int(pred_result.boxes.conf.argmax())
        pred = pred_result.keypoints.xy[ix].detach().cpu().numpy()
        d2 = ((gt - pred) ** 2).sum(axis=1)
        record.update(pred=pred,
                      oks=float(np.exp(-d2 / (2 * area * (2 * SIGMAS) ** 2)).mean()),
                      err_px=np.sqrt(d2))
    records.append(record)
assert len(records) == 53
missed = sum(r["pred"] is None for r in records)
worst = sorted(records, key=lambda r: r["oks"])[:6]

import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, rec in zip(axes.flat, worst):
    image = cv2.cvtColor(cv2.imread(str(rec["path"])), cv2.COLOR_BGR2RGB)
    ax.imshow(image)
    ax.scatter(rec["gt"][:, 0], rec["gt"][:, 1], s=38, c="red", marker="o", label="GT")
    if rec["pred"] is not None:
        ax.scatter(rec["pred"][:, 0], rec["pred"][:, 1], s=48, c="cyan", marker="x", label="Dự đoán")
        for (gx, gy), (px, py) in zip(rec["gt"], rec["pred"]):
            ax.plot([gx, px], [gy, py], color="yellow", linewidth=0.8, alpha=0.8)
    ax.set_title(f"{rec['path'].name} — OKS {rec['oks']:.3f}")
    ax.axis("off")
axes.flat[0].legend(loc="lower right")
fig.suptitle("6 ảnh validation có OKS thấp nhất — đỏ: GT · cyan: dự đoán · vàng: sai số", fontsize=14)
fig.tight_layout()
SUBMISSION = Path("submission")
(SUBMISSION / "autolabel").mkdir(parents=True, exist_ok=True)
Q11_IMAGE = SUBMISSION / "q11_val_samples.png"
fig.savefig(Q11_IMAGE, dpi=140, bbox_inches="tight")
plt.show()
plt.close(fig)

# Tạo Q11 có số liệu trực tiếp từ ảnh của lần train 40 epoch này.
valid = [r for r in worst if r["pred"] is not None]
if valid:
    all_worst_errors = np.stack([r["err_px"] for r in valid])
    mean_by_kpt = np.nanmean(all_worst_errors, axis=0)
    hardest = np.argsort(mean_by_kpt)[-3:][::-1]
    hardest_desc = ", ".join(
        f"{KPT_NAMES[int(i)]} ({mean_by_kpt[i]:.0f} px trung bình trong nhóm sáu ảnh)"
        for i in hardest
    )
    details = []
    for rec in valid[:2]:
        k = int(np.nanargmax(rec["err_px"]))
        details.append(f"{rec['path'].name} (OKS {rec['oks']:.3f}) có lỗi lớn nhất ở {KPT_NAMES[k]}: {rec['err_px'][k]:.0f} px")
    Q11 = (
        f"Trên checkpoint sau đủ {completed_epochs} epoch, sáu ảnh val thấp nhất có OKS "
        f"{worst[-1]['oks']:.3f} trở xuống; {details[0]}, còn {details[1]}. "
        f"Trong nhóm này các keypoint lệch nhiều nhất là {hardest_desc}. "
        f"Ảnh đỏ là nhãn thật, cyan là dự đoán và vạch vàng nối sai số trong "
        f"`submission/q11_val_samples.png`. Lỗi chân/điểm xa thân thường cần thêm ảnh có tư thế bước hoặc che khuất tương tự và rà soát nhãn trái-phải; "
        f"nếu nhiều keypoint cùng lệch, cần kiểm tra chất lượng nhãn và đa dạng vị trí/kích thước của hổ trong train."
    )
else:
    Q11 = f"Model không phát hiện được hổ ở các ảnh val tệ nhất; miss={missed}/53. Cần kiểm tra ảnh đối chiếu `submission/q11_val_samples.png`, bổ sung ảnh/nhãn khó và điều chỉnh dữ liệu huấn luyện."
print("Q11 cập nhật theo lần chạy này:\n" + Q11)


In [ ]:
# Cập nhật báo cáo cũ mà không làm mất Q1-Q10/Q12, lưu kết quả train thật và đóng gói.
BASE_REPORT["student"] = {"name": "Trần Thị Như Ý", "student_id": "2A202602372"}
BASE_REPORT = json.loads('{"progress":{"box_iou":"ok","nms":"ok","batched_nms":"ok","average_precision":"ok","mask_iou":"ok","polygon_to_mask":"ok","mask_to_yolo_seg":"ok","autolabel":"ok","oks":"ok","joint_angle":"ok","FLIP_IDX":"ok"},"latency":[{"Cấu hình":"one-to-many + NMS, conf 0.25","preprocess (ms)":1.75,"inference (ms)":33.08,"postprocess (ms)":0.82,"số box":5},{"Cấu hình":"one-to-many + NMS, conf 0.001","preprocess (ms)":1.76,"inference (ms)":33.15,"postprocess (ms)":1.16,"số box":203},{"Cấu hình":"one-to-one NMS-free, conf 0.25","preprocess (ms)":2.08,"inference (ms)":40.07,"postprocess (ms)":0.33,"số box":5},{"Cấu hình":"one-to-one NMS-free, conf 0.001","preprocess (ms)":2.29,"inference (ms)":50.72,"postprocess (ms)":0.37,"số box":204}],"result_4b":{"flip_idx":"giải phẫu","epochs":3,"imgsz":640,"Box mAP50-95":0.41857352505390744,"Pose mAP50":0.0,"Pose mAP50-95":0.0,"train (phút)":8.9},"result_4c":null,"answers":{"Q1":"Faster R-CNN trả về dictionary theo ảnh với `boxes` (N,4), `labels` (N) và `scores` (N); mỗi hàng ứng với một detection. YOLO26 trả về tensor dự đoán dày trước bước lọc, với 4 tọa độ và 80 điểm lớp trên từng vị trí; ở 640×640 có 8400 vị trí (3 tầng feature), còn ảnh bus letterbox thành 640×480 có 6300. Faster R-CNN đã chạy NMS trong hai stage; tensor thô của YOLO cần hậu xử lý.","Q2":"Head one-to-many phát nhiều ứng viên cho cùng vật thể nên NMS phải loại box trùng; conf thấp giữ nhiều ứng viên hơn và làm NMS tốn hơn. Head one-to-one được huấn luyện để mỗi vật thể có một dự đoán, nên có thể bỏ NMS. Conf 0.001 thường cho nhiều box hơn conf 0.25; độ trễ pipeline cần đo thực nghiệm trên cùng ảnh/phần cứng.","Q3":"NMS của tôi giữ các box theo score giảm dần và chỉ loại box cùng lớp có IoU lớn hơn ngưỡng. Với bus.jpg, ngưỡng conf 0.25 cho phép đếm 4 người và 1 xe buýt; kết quả NMS thủ công khớp Ultralytics (5 box). Khi conf hạ xuống 0.001, nhiều dự đoán yếu được giữ lại và cần NMS mạnh hơn.","Q4":"Semantic segmentation gán một lớp cho mỗi pixel, không gắn ID riêng cho từng người. Vì vậy các vùng của nhiều người có thể liền nhau hoặc một người có thể thành nhiều thành phần liên thông; class pixel cũng có thể nhầm, chẳng hạn vùng xe buýt bị gán train. Instance segmentation tách từng đối tượng và trả mask riêng, phù hợp để đếm và ghép với box.","Q5":"Mask IoU = diện tích giao chia diện tích hợp; hai hình vuông 50×50 lệch 25 pixel có IoU 625/4375 ≈ 0.1429. Polygon được chuẩn hóa bằng x/W và y/H để ghi đúng một dòng YOLO-seg. Mỗi prompt điểm cho SAM có thể bao phủ đối tượng khác nhau: điểm trong thân thường tạo mask người gần đầy đủ, còn điểm ở rìa có thể chỉ chọn mảnh nhỏ.","Q6":"Auto-label chạy detector YOLO26-seg để lấy box/prompt rồi dùng SAM 2.1 tạo mask; contour lớn nhất được xấp xỉ thành polygon và ghi theo định dạng YOLO-seg với tọa độ trong [0,1]. `bus.txt` có một dòng cho mỗi mask được giữ (kỳ vọng 5 dòng trên ảnh bus mẫu); cần vẽ lại polygon để kiểm tra biên trước khi dùng làm nhãn huấn luyện.","Q7":"Keypoint R-CNN biểu diễn mỗi điểm bằng (x,y,confidence), nên đầu ra hai người có dạng (2,17,3); YOLO26-pose tách tọa độ và confidence, tọa độ có dạng (2,17,2). Ngưỡng confidence giúp bỏ điểm không đáng tin. Khi đặt ngưỡng rất thấp, mô hình vẫn vẽ điểm dự đoán ngoài khung ảnh; tọa độ đó không nên được xem là quan sát hợp lệ.","Q8":"Theo công thức OKS và σ mắt COCO = 0.026, mức tương đồng của mắt giảm dưới 0.5 ở sai số xấp xỉ 3 px với người 40×80 và 19 px với người 250×500 (vì diện tích đối tượng lớn làm cùng sai số pixel bị phạt nhẹ hơn). σ mắt nhỏ hơn σ hông vì vị trí mắt được gán chính xác hơn, còn hông có biến thiên/che khuất lớn hơn. Với người cao 80 px, vài pixel sai ở mắt đã làm OKS giảm đáng kể; camera trên cao cần độ phân giải tốt, ngưỡng tin cậy và có thể dùng σ phù hợp dữ liệu.","Q9":"Luật nghiêng >60° có thể báo nhầm khi người cúi nhặt đồ, ngồi hoặc nghiêng người; có thể bỏ sót khi người ngã nhưng tay/chân còn dựng hoặc thân bị che. Ảnh xoay làm thay đổi phân bố hướng và kích thước so với ảnh huấn luyện, nên detector pose có thể bỏ người. Hệ thống thực tế cần theo dõi nhiều khung hình, lọc nhiễu theo thời gian, kiểm tra vận tốc/tư thế và vùng nguy hiểm, xử lý che khuất, đặt ngưỡng theo camera, rồi xác nhận cảnh báo để giảm báo động giả.","Q10":"Trên split này, giữ flip_idx đồng nhất có thể không làm mAP val giảm rõ vì train và val đều chỉ có hổ quay phải; phép lật augmentation tạo nhãn nhất quán theo quy ước chỉ số đang dùng. Bug xuất hiện khi triển khai gặp hổ quay trái hoặc ảnh gương: nhãn giải phẫu trái/phải bị đảo, làm keypoint chân sai dù hình học nhìn có vẻ hợp lý. Val cần có cả hai hướng và ảnh lật gương để bộc lộ vấn đề.","Q11":"Trên lần chạy CPU fallback 3 epoch, Pose mAP50 = 0.0 nên các dự đoán còn lỗi nặng. Trong nhóm sáu ảnh OKS thấp nhất, Frame_28 có mũi dự đoán lệch khoảng 749 px và Frame_122 lệch khoảng 685 px; ảnh đối chiếu cho thấy nhiều keypoint bị dồn ra ngoài mép trái thay vì phủ lên thân hổ. Ở Frame_203 và Frame_137, cả cụm keypoint cũng co lệch về mép trái, trong khi nhãn thật trải dọc toàn thân; đây là lỗi định vị/tỉ lệ toàn pose, không thể khắc phục bằng đổi ngưỡng. Cách khắc phục chính là chạy đủ 40 epoch trên GPU rồi kiểm tra lại pose mAP và sáu ảnh val xấu nhất; nếu vẫn còn lệch, kiểm tra phép biến đổi tọa độ box/keypoint và bổ sung ảnh đa dạng hướng, vị trí. Ảnh chấm đỏ là GT, ô cyan là dự đoán: `submission/q11_val_samples.png`.","Q12":"Một σ đồng nhất cho 12 điểm là giả định tiện dụng nhưng không biểu diễn việc mũi thường định vị rõ hơn gốc đuôi hoặc các chân bị che. Tôi sẽ thu thập nhiều nhãn độc lập cho mỗi keypoint, đo độ lệch giữa người gán nhãn sau khi chuẩn hóa theo kích thước đối tượng, rồi chọn σ theo độ biến thiên thực nghiệm; đồng thời kiểm tra trên tập val đa dạng tư thế, hướng quay và mức che khuất. Có thể báo thêm AP theo từng nhóm keypoint để metric tổng không che lỗi ở điểm quan trọng."},"env":{"device":"cpu","torch":"2.14.0+cpu","ultralytics":"8.4.171"},"audit":{"notebook_run_all":"completed_without_cell_errors","executed_code_cells":53,"total_code_cells":53,"core_requirement_status":"incomplete: 4B requires 40 GPU epochs; current environment is CPU-only and fallback ran 3 epochs","run_4b_pose_map50":0.0,"run_4b_pose_map50_95":0.0,"q11_visual":"submission/q11_val_samples.png","lms_public_repo_submission":"manual external step"},"remaining_requirements":["Run 4B with GPU for 40 epochs, then regenerate Q11 analysis and mAP outputs","Publish the completed notebook and submission folder to a public GitHub repository and submit its URL in LMS"]}')
(BASE_REPORT.setdefault("answers", {}))["Q11"] = Q11
BASE_REPORT["result_4b"] = RESULT_4B
BASE_REPORT["remaining_requirements"] = [
    "Push the completed notebook and submission folder to a public GitHub repository",
    "Submit the public repository URL in LMS",
]
BASE_REPORT["audit"] = {
    "gpu_training": "completed",
    "completed_epochs": completed_epochs,
    "requested_epochs": 40,
    "device": torch.cuda.get_device_name(0),
    "val_images": len(val_imgs),
    "best_checkpoint": str(SAVE_DIR / "weights" / "best.pt"),
    "q11_image": str(Q11_IMAGE),
    "github_and_lms": "GitHub main branch pushed; LMS URL submission pending",
    "public_repository": "https://github.com/nhuY02/Track04-Day18-2D-perception-detection-segmentation-keypoints",
}
BASE_REPORT.setdefault("progress", {})["fine_tune_4B"] = "ok_40_gpu_epochs"
BASE_REPORT["env"] = {
    "device": torch.cuda.get_device_name(0), "torch": torch.__version__,
    "torch_cuda": torch.version.cuda, "ultralytics": ultralytics.__version__,
}
SUBMISSION.mkdir(exist_ok=True)
(SUBMISSION / "autolabel" / "bus.txt").write_bytes(base64.b64decode('NSAwLjAxMjM0NiAwLjM0ODE0OCAwLjAxNjA0OSAwLjQzMTQ4MSAwLjAyMzQ1NyAwLjQxNzU5MyAwLjAzMDg2NCAwLjQzNzAzNyAwLjAxMzU4MCAwLjQzODg4OSAwLjAzMDg2NCAwLjQ1Mjc3OCAwLjAyMDk4OCAwLjU0NDQ0NCAwLjA2MDQ5NCAwLjYyMTI5NiAwLjA4MDI0NyAwLjYyNDA3NCAwLjA2NzkwMSAwLjUyODcwNCAwLjA4Mzk1MSAwLjQ0NjI5NiAwLjEwOTg3NyAwLjQzNTE4NSAwLjEyOTYzMCAwLjM3NTkyNiAwLjE2NDE5OCAwLjM2NTc0MSAwLjE5NjI5NiAwLjM3Nzc3OCAwLjE5MTM1OCAwLjQyNjg1MiAwLjI0OTM4MyAwLjUxNDgxNSAwLjI0ODE0OCAwLjU0MTY2NyAwLjIyODM5NSAwLjU1NTU1NiAwLjI0ODE0OCAwLjYzMDU1NiAwLjIzMDg2NCAwLjY3MDM3MCAwLjI5NTA2MiAwLjY2NDgxNSAwLjI3NDA3NCAwLjY1MzcwNCAwLjI5MjU5MyAwLjQ2Mjk2MyAwLjMzMzMzMyAwLjQzNTE4NSAwLjMyNDY5MSAwLjM4MTQ4MSAwLjM3NDA3NCAwLjM4MzMzMyAwLjM3Nzc3OCAwLjQzNTE4NSAwLjQwMzcwNCAwLjQ0NzIyMiAwLjQyNTkyNiAwLjUxMzg4OSAwLjQwMjQ2OSAwLjUzMjQwNyAwLjQxOTc1MyAwLjY3Nzc3OCAwLjQ1NDMyMSAwLjY2OTQ0NCAwLjQ2MDQ5NCAwLjY0ODE0OCAwLjUxOTc1MyAwLjY0MDc0MSAwLjUzMzMzMyAwLjYwOTI1OSAwLjU1OTI1OSAwLjYwODMzMyAwLjU4MDI0NyAwLjYzOTgxNSAwLjgwMDAwMCAwLjYwODMzMyAwLjgxODUxOSAwLjU3OTYzMCAwLjg0NDQ0NCAwLjU3ODcwNCAwLjg1NTU1NiAwLjYyMzE0OCAwLjkxODUxOSAwLjYyMTI5NiAwLjkzMzMzMyAwLjU2Mzg4OSAwLjg4NTE4NSAwLjUwOTI1OSAwLjk1MDYxNyAwLjQ5ODE0OCAwLjk3OTAxMiAwLjQyNzc3OCAwLjk3MTYwNSAwLjM4NDI1OSAwLjk5MTM1OCAwLjM1ODMzMyAwLjk3NjU0MyAwLjI5NzIyMiAwLjg1Njc5MCAwLjI0NTM3MCAwLjI5NzUzMSAwLjIxMzg4OSAwLjEwNzQwNyAwLjIzOTgxNSAwLjA2MDQ5NCAwLjI4MzMzMyAwLjA0MDc0MSAwLjM0NTM3MA0KMCAwLjM1Njc5MCAwLjM3Njg1MiAwLjMzNzAzNyAwLjM3Nzc3OCAwLjMyNDY5MSAwLjM4NzAzNyAwLjMyNDY5MSAwLjQxMjAzNyAwLjMzNzAzNyAwLjQzNDI1OSAwLjMwMTIzNSAwLjQ2Mjk2MyAwLjI5MTM1OCAwLjQ3ODcwNCAwLjI5MzgyNyAwLjUwNzQwNyAwLjI5MDEyMyAwLjU0MTY2NyAwLjI5NjI5NiAwLjU1NzQwNyAwLjI4Mzk1MSAwLjU5MjU5MyAwLjI3Nzc3OCAwLjY0NTM3MCAwLjMwMDAwMCAwLjY1Mjc3OCAwLjMwMDAwMCAwLjY4ODg4OSAwLjMxMjM0NiAwLjc1NDYzMCAwLjMxNzI4NCAwLjc2MDE4NSAwLjMxNDgxNSAwLjc4MzMzMyAwLjMxOTc1MyAwLjc5MDc0MSAwLjMzNDU2OCAwLjc5NTM3MCAwLjM1MDYxNyAwLjc5NTM3MCAwLjM1OTI1OSAwLjc5MTY2NyAwLjM0NjkxNCAwLjc2MDE4NSAwLjM1NTU1NiAwLjc1MzcwNCAwLjM2NDE5OCAwLjc1NTU1NiAwLjM1MTg1MiAwLjczNzAzNyAwLjM1MzA4NiAwLjcyMjIyMiAwLjQwOTg3NyAwLjY3Nzc3OCAwLjQxODUxOSAwLjY2Mzg4OSAwLjQxMzU4MCAwLjY0NjI5NiAwLjQxNzI4NCAwLjYzNDI1OSAwLjM5NzUzMSAwLjUyODcwNCAwLjQxNzI4NCAwLjUyMDM3MCAwLjQyMzQ1NyAwLjUwOTI1OSAwLjQwNzQwNyAwLjQ1MzcwNCAwLjM3NDA3NCAwLjQzODg4OSAwLjM4MTQ4MSAwLjQxNTc0MSAwLjM4MDI0NyAwLjQwMTg1MiAwLjM3Mjg0MCAwLjM4NDI1OQ0KMCAwLjE2MDQ5NCAwLjM2ODUxOSAwLjEzNzAzNyAwLjM3NTAwMCAwLjEyNzE2MCAwLjM4NDI1OSAwLjEyNTkyNiAwLjQxNDgxNSAwLjExODUxOSAwLjQzMTQ4MSAwLjA4MDI0NyAwLjQ1NTU1NiAwLjA3NTMwOSAwLjQ2ODUxOSAwLjA3NDA3NCAwLjYzODg4OSAwLjA5MDEyMyAwLjY1Mjc3OCAwLjA4ODg4OSAwLjcwMDkyNiAwLjA5ODc2NSAwLjcyOTYzMCAwLjA4NTE4NSAwLjc0NzIyMiAwLjA3NDA3NCAwLjc1MTg1MiAwLjA2Mjk2MyAwLjgxNzU5MyAwLjA3OTAxMiAwLjgyOTYzMCAwLjEwMTIzNSAwLjgzNDI1OSAwLjEyMzQ1NyAwLjgzMzMzMyAwLjEzMDg2NCAwLjgyNTkyNiAwLjEwNjE3MyAwLjc5NDQ0NCAwLjExMzU4MCAwLjczNzAzNyAwLjEzMjA5OSAwLjcwNzQwNyAwLjE0ODE0OCAwLjY1MzcwNCAwLjE2MDQ5NCAwLjY1ODMzMyAwLjE4NTE4NSAwLjY5OTA3NCAwLjIxNzI4NCAwLjc5NDQ0NCAwLjIxODUxOSAwLjgzMDU1NiAwLjI2OTEzNiAwLjgyNzc3OCAwLjMwMTIzNSAwLjgxNzU5MyAwLjI5NzUzMSAwLjgxMTExMSAwLjI2NjY2NyAwLjc5OTA3NCAwLjI0ODE0OCAwLjc4NTE4NSAwLjIyMDk4OCAwLjY0MTY2NyAwLjIzNzAzNyAwLjYzNzk2MyAwLjIzNTgwMiAwLjYwMjc3OCAwLjIyMjIyMiAwLjU1NTU1NiAwLjI0NTY3OSAwLjU0MDc0MSAwLjI0NjkxNCAwLjUxNTc0MSAwLjIyOTYzMCAwLjQ5MzUxOSAwLjIyMzQ1NyAwLjQ2ODUxOSAwLjE4NjQyMCAwLjQyOTYzMCAwLjE5MzgyNyAwLjM4MDU1Ng0KMCAwLjk5ODc2NSAwLjM2NDgxNSAwLjk4MTQ4MSAwLjM3MzE0OCAwLjk3NjU0MyAwLjM4MDU1NiAwLjk3NTMwOSAwLjQwNDYzMCAwLjk4NTE4NSAwLjQyNjg1MiAwLjk3MDM3MCAwLjQ1MDAwMCAwLjk2MDQ5NCAwLjQ1NzQwNyAwLjk1MzA4NiAwLjUwMDAwMCAwLjk0ODE0OCAwLjUwNzQwNyAwLjkzOTUwNiAwLjUxMDE4NSAwLjkwODY0MiAwLjUwMjc3OCAwLjg5MTM1OCAwLjUxMTExMSAwLjg5NzUzMSAwLjUyNzc3OCAwLjkxMzU4MCAwLjUzMTQ4MiAwLjkxODUxOSAwLjU0NDQ0NCAwLjkzMzMzMyAwLjU0NzIyMiAwLjkzODI3MiAwLjU1NDYzMCAwLjkzNTgwMiAwLjU4NjExMSAwLjkyNDY5MSAwLjYzMTQ4MSAwLjkzNzAzNyAwLjYzODg4OSAwLjkxNzI4NCAwLjY3Nzc3OCAwLjkxMzU4MCAwLjcwMDkyNiAwLjg4MTQ4MSAwLjc2Mjk2MyAwLjg3MDM3MCAwLjc2ODUxOSAwLjg1Njc5MCAwLjc2ODUxOSAwLjgzODI3MiAwLjc2Mjk2MyAwLjgyNzE2MCAwLjc2NTc0MSAwLjgyOTYzMCAwLjc3Nzc3OCAwLjgzNzAzNyAwLjc4MzMzMyAwLjg4MjcxNiAwLjgwMTg1MiAwLjkwMDAwMCAwLjgxMjk2MyAwLjkwOTg3NyAwLjgxMDE4NSAwLjkyMDk4OCAwLjc2ODUxOSAwLjkzNzAzNyAwLjc0ODE0OCAwLjk2NzkwMSAwLjY4OTgxNSAwLjk5ODc2NSAwLjY1NTU1Ng0KMCAwLjAwMDAwMCAwLjUxMzg4OSAwLjAwMDAwMCAwLjgwNjQ4MSAwLjAwNzQwNyAwLjgwNjQ4MSAwLjAxMjM0NiAwLjgwMjc3OCAwLjAyOTYzMCAwLjgwMjc3OCAwLjAzODI3MiAwLjc5OTA3NCAwLjAzMDg2NCAwLjc5MTY2NyAwLjAyMjIyMiAwLjc3NTAwMCAwLjAyOTYzMCAwLjcyODcwNCAwLjAzMDg2NCAwLjcwMTg1MiAwLjAzODI3MiAwLjY5NTM3MCAwLjA1NDMyMSAwLjY5NDQ0NCAwLjA3NTMwOSAwLjY4MjQwNyAwLjA3NTMwOSAwLjY2NzU5MyAwLjA0MTk3NSAwLjYwNzQwNyAwLjAyODM5NSAwLjU2OTQ0NCAwLjAxNzI4NCAwLjU0ODE0OCAwLjAxMTExMSAwLjUyMTI5Ng0K'))
(SUBMISSION / "4b_results.csv").write_bytes(RESULTS_CSV.read_bytes())
(SUBMISSION / "4b_gpu_result.json").write_text(json.dumps({
    "student": BASE_REPORT.get("student"), "result_4b": RESULT_4B,
    "answer_Q11": Q11, "audit": BASE_REPORT["audit"],
}, ensure_ascii=False, indent=2), encoding="utf-8")
(SUBMISSION / "ket_qua.json").write_text(json.dumps(BASE_REPORT, ensure_ascii=False, indent=2), encoding="utf-8")
shutil.make_archive("submission", "zip", root_dir=".", base_dir="submission")
print("Đã tạo submission/ket_qua.json, submission/q11_val_samples.png, submission/4b_results.csv và submission.zip")
print(json.dumps(RESULT_4B, ensure_ascii=False, indent=2))


In [ ]:
from google.colab import files
files.download("submission.zip")

## Sau khi chạy xong

Tải `submission.zip` về. Thay file `submission/ket_qua.json`, `submission/autolabel/bus.txt` và ảnh `submission/q11_val_samples.png` trong bản project của bạn bằng các file trong zip. Giữ `4b_results.csv` làm bằng chứng log 40 epoch.